In [1]:
import xarray as xr
import pandas as pd

In [3]:
data = xr.open_zarr(
    "/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_OM4_5daily_v0.2.1"
)
repeats = 100

In [4]:
# Choose year 1996
data_1996 = data.sel(time=slice("1996-01-01", "1996-12-31"))

# Print first and last time steps
print(data_1996.time[0].values, data_1996.time[-1].values)

print(data_1996)


1996-01-03 12:00:00 1996-12-28 12:00:00
<xarray.Dataset>
Dimensions:            (y: 180, x: 360, lev: 19, time: 73, y_b: 181, x_b: 361)
Coordinates:
    areacello          (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz                 (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction     (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time               (time) object 1996-01-03 12:00:00 ... 1996-12-28 12:00:00
    wetmask            (lev, y, x) bool dask.array<chunksize=(10, 90, 36

In [5]:
# Generate a new time coordinate for repeats years, repeating 1996
new_time = pd.date_range(start=str(data_1996.time[0].values), periods=repeats * len(data_1996.time), freq="5D")

In [6]:
# Assert first year are the same dates 
for i in range(len(data_1996.time)):
    print(i , end=" ")
    assert str(new_time[i]) == str(data_1996.time[i].values)

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 

In [7]:
# Repeat the data for each day to match the new time length
repeated_data = xr.concat([data_1996] * repeats, dim="time")
repeated_data['time'] = new_time  # Update the time coordinate

In [8]:
from dask.diagnostics import ProgressBar

In [9]:
# Save data
with ProgressBar():
    repeated_data.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_OM4_5daily_v0.2.1_100_years")

[########################################] | 100% Completed | 11m 34s


In [10]:
with ProgressBar():
    ds_mean = data_1996.mean().compute()
    
with ProgressBar():
    ds_mean.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_OM4_5daily_v0.2.1_100_years_means")

[########################################] | 100% Completed | 12.34 s


In [11]:
with ProgressBar():
    ds_mean = data_1996.std().compute()

with ProgressBar():
    ds_mean.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/3D_data_OM4_5daily_v0.2.1_100_years_stds")

[########################################] | 100% Completed | 15.20 s
